In [67]:
import pandas as pd
import umap
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler


In [68]:
# 1. Load the datasets
print("Loading datasets...")
# Replace 'sample_id' with the actual name of the sample column in your methylation data if different
clin_df = pd.read_csv('../data/processed/clinical_hnsc_all_sb.csv')

Loading datasets...


In [69]:
meth_df = pd.read_csv('../data/processed/methylation_hnsc_all.csv', index_col=0)
meth_df

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [ ]:
# Extract sample IDs from the index
meth_samples = meth_df.index.values

# Extract the raw methylation features (all columns)
meth_features = meth_df.values

# (Optional but highly recommended) Scale the data before UMAP and Clustering
print("Scaling data...")
scaler = StandardScaler()
meth_features_scaled = scaler.fit_transform(meth_features)

# 2. Dimensionality Reduction (UMAP from 406601 to 2)
print("Running UMAP (n=2)...")
# Note: Using the scaled features here, but you can change to meth_features if you prefer raw
reducer = umap.UMAP(n_components=3, random_state=42)
meth_umap = reducer.fit_transform(meth_features_scaled)

# 3. Clustering 

N_CLUSTERS = 2

# Feature 1: K-means (K=6) on UMAP data
print("Running K-Means on UMAP data...")
kmeans_umap = KMeans(n_clusters=N_CLUSTERS, random_state=42)
kmeans_umap_labels = kmeans_umap.fit_predict(meth_umap)

# Feature 2: K-means (K=6) on Raw data
print("Running K-Means on Raw data (This may take a while)...")
kmeans_raw = KMeans(n_clusters=N_CLUSTERS, random_state=42)
kmeans_raw_labels = kmeans_raw.fit_predict(meth_features_scaled)

Scaling data...
Running UMAP (n=2)...


/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Running K-Means on UMAP data...
Running K-Means on Raw data (This may take a while)...


/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [ ]:
# Feature 3: DBSCAN on UMAP data
print("Running DBSCAN on UMAP data...")
# Note: You will need to tune eps and min_samples to find your desired 6 clusters
dbscan_umap = DBSCAN(eps=0.5, min_samples=5) 
dbscan_umap_labels = dbscan_umap.fit_predict(meth_umap)

# Feature 4: DBSCAN on Raw data
print("Running DBSCAN on Raw data (This will require high memory)...")
# Note: Distances in high dimensions are very large; eps usually needs to be significantly higher here.
dbscan_raw = DBSCAN(eps=5.0, min_samples=5) 
dbscan_raw_labels = dbscan_raw.fit_predict(meth_features_scaled)

# Feature 5: Hierarchical (Agglomerative) on UMAP data
print(f"Running Hierarchical Clustering (K={N_CLUSTERS}) on UMAP data...")
# 'ward' linkage minimizes the variance of the clusters being merged
hierarchical_umap = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage='ward')
hierarchical_umap_labels = hierarchical_umap.fit_predict(meth_umap)

# Feature 6: Hierarchical (Agglomerative) on Raw data
print(f"Running Hierarchical Clustering (K={N_CLUSTERS}) on Raw data...")
hierarchical_raw = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage='ward')
hierarchical_raw_labels = hierarchical_raw.fit_predict(meth_features_scaled)


Running DBSCAN on UMAP data...
Running DBSCAN on Raw data (This will require high memory)...
Running Hierarchical Clustering (K=2) on UMAP data...
Running Hierarchical Clustering (K=2) on Raw data...


In [ ]:
 #4. Compile the new features into a DataFrame
print("Compiling results...")
cluster_results = pd.DataFrame({
    'sample': meth_samples,
    'kmeans_umap_2_k6': kmeans_umap_labels,
    'kmeans_raw_k6': kmeans_raw_labels,
    'dbscan_umap_2': dbscan_umap_labels,
    'dbscan_raw': dbscan_raw_labels,
    'hierarchical_umap_2': hierarchical_umap_labels,
    'hierarchical_raw': hierarchical_raw_labels
})

# 5. Merge with the clinical dataset
print("Merging with clinical data...")
# Merging on 'sample' column. Using an inner or left join to ensure alignment
final_df = pd.merge(clin_df, cluster_results, on='sample', how='left')

# 6. Save the new dataset
output_file = '../data/processed/clinical_hnsc_all_sb_clustered.csv'

Compiling results...
Merging with clinical data...


In [ ]:
final_df

,sample,sample_type.samples,tissue_type.samples,primary_site,ajcc_pathologic_stage.diagnoses,alcohol_history.exposures,gender.demographic,race.demographic,vital_status.demographic,tissue_source_site_id.tissue_source_site,...,Subtype_mRNA,Subtype_miRNA,Subtype_other,Subtype_protein,kmeans_umap_2_k6,kmeans_raw_k6,dbscan_umap_2,dbscan_raw,hierarchical_umap_2,hierarchical_raw
0,TCGA-BB-4227-01A,Primary Tumor,Tumor,Hypopharynx,Stage IVA,Yes,male,white,Alive,a7da35ad-149e-5014-932b-37f188536bf8,...,NaN,NaN,NaN,NaN,0.0,0.0,1.0,-1.0,0.0,0.0
1,TCGA-CR-6478-01A,Primary Tumor,Tumor,Tonsil,Unknown,Yes,female,white,Dead,041ae4a7-3724-5587-82a9-8e09d3ee3cb8,...,Mesenchymal,3.0,Paradigma.4,NaN,0.0,1.0,1.0,-1.0,0.0,0.0
2,TCGA-BB-4228-01A,Primary Tumor,Tumor,Base of tongue,Stage III,Yes,male,white,Alive,a7da35ad-149e-5014-932b-37f188536bf8,...,Atypical,3.0,Paradigma.5,NaN,0.0,1.0,1.0,-1.0,0.0,0.0
3,TCGA-BA-4075-01A,Primary Tumor,Tumor,Other and unspecified parts of tongue,Stage III,Yes,male,black or african american,Dead,278e36e9-7faf-5043-ba90-2a4145ad0226,...,NaN,NaN,NaN,NaN,0.0,1.0,1.0,-1.0,0.0,0.0
4,TCGA-CX-7082-01A,Primary Tumor,Tumor,"Other and ill-defined sites in lip, oral cavit...",Stage IVA,Yes,male,white,Dead,5184bb4b-3075-56fb-886f-7895a42e156b,...,Classical,1.0,Paradigma.2,NaN,0.0,0.0,1.0,-1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599,TCGA-CV-7242-11A,Solid Tissue Normal,Normal,Larynx,Stage II,Yes,female,white,Alive,e0529816-35d4-5963-93c3-72c127e4e372,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
600,TCGA-TN-A7HL-01A,Primary Tumor,Tumor,Hypopharynx,Stage IVA,Yes,male,white,Alive,0880c76d-8168-580c-82c0-aeb6ba104840,...,NaN,NaN,NaN,NaN,0.0,1.0,1.0,-1.0,0.0,0.0
601,TCGA-CV-A6JZ-01A,Primary Tumor,Tumor,Other and unspecified parts of mouth,Stage II,Yes,male,white,Alive,e0529816-35d4-5963-93c3-72c127e4e372,...,NaN,NaN,NaN,NaN,1.0,1.0,1.0,-1.0,0.0,0.0
602,TCGA-CV-5444-11A,Solid Tissue Normal,Normal,Larynx,Stage IVA,Yes,male,black or african american,Alive,e0529816-35d4-5963-93c3-72c127e4e372,...,NaN,NaN,NaN,NaN,1.0,1.0,2.0,-1.0,1.0,0.0


In [ ]:
final_df.to_csv(output_file, index=False)
print(f"Process complete! Saved new dataset to {output_file}")

Process complete! Saved new dataset to ../data/processed/clinical_hnsc_all_sb_clustered.csv


In [ ]:
# import pandas as pd
# from sklearn.metrics import adjusted_rand_score

# # 1. Load the dataset (if you are running this in a new script)
# # final_df = pd.read_csv('clinical_hnsc_all_sb_clustered.csv')

# # 2. Drop rows where the ground truth is missing
# # ARI cannot evaluate samples that do not have a known immune subtype
# #valid_df = final_df.dropna(subset=['immune_subtypes']).copy()
# #valid_df = final_df.dropna(subset=['Subtype_mRNA']).copy()
# valid_df = final_df.dropna(subset=['tissue_type.samples']).copy()

# # Extract the true labels
# #true_labels = valid_df['immune_subtypes']
# #true_labels = valid_df['Subtype_mRNA']
# true_labels = valid_df['tissue_type.samples']

# # 3. Define the clustering columns to evaluate
# cluster_columns = [
#     'kmeans_umap_50_k6',
#     'kmeans_raw_k6',
#     'dbscan_umap_50',
#     'dbscan_raw'
# ]

# # 4. Compute and print the ARI for each feature
# print("Adjusted Rand Index (ARI) Scores:")
# print("-" * 40)

# for col in cluster_columns:
#     pred_labels = valid_df[col]
#     ari_score = adjusted_rand_score(true_labels, pred_labels)
#     print(f"{col:<20} : {ari_score:.4f}")

In [ ]:
import pandas as pd
from sklearn.metrics import adjusted_rand_score

# 1. Load the dataset (if you are running this in a new script)
# final_df = pd.read_csv('clinical_hnsc_all_sb_clustered.csv')

# 2. Define columns and drop ALL rows containing NaNs in these specific columns
cluster_columns = [
    'kmeans_umap_2_k6',
    'kmeans_raw_k6',
    'dbscan_umap_2',
    'dbscan_raw',
    'hierarchical_umap_2',
    'hierarchical_raw'
]
target_column = 'immune_subtypes'  # Change this to the actual name of your ground truth column

columns_to_check = [target_column] + cluster_columns
valid_df = final_df.dropna(subset=columns_to_check).copy()

# Extract the true labels
true_labels = valid_df[target_column]

# 3. Compute and print the ARI for each feature
print(f"Adjusted Rand Index (ARI) Scores (Evaluated on {len(valid_df)} samples):")
print("-" * 55)

for col in cluster_columns:
    pred_labels = valid_df[col]
    ari_score = adjusted_rand_score(true_labels, pred_labels)
    print(f"{col:<25} : {ari_score:.4f}")

Adjusted Rand Index (ARI) Scores (Evaluated on 514 samples):
-------------------------------------------------------
kmeans_umap_2_k6          : 0.0073
kmeans_raw_k6             : 0.0652
dbscan_umap_2             : 0.0423
dbscan_raw                : 0.0000
hierarchical_umap_2       : 0.0044
hierarchical_raw          : 0.0476


In [ ]:
for col in valid_df.columns:
    print(f"--- {col} ---")
    print(valid_df[col].value_counts())
    print("\n")

--- sample ---
sample
TCGA-BB-4227-01A    1
TCGA-IQ-A61G-01A    1
TCGA-WA-A7H4-01A    1
TCGA-CR-7397-01A    1
TCGA-MZ-A7D7-01A    1
                   ..
TCGA-CV-7413-01A    1
TCGA-IQ-A6SG-01A    1
TCGA-CV-A6JU-01A    1
TCGA-BA-6869-01A    1
TCGA-CV-5444-01A    1
Name: count, Length: 514, dtype: int64


--- sample_type.samples ---
sample_type.samples
Primary Tumor    514
Name: count, dtype: int64


--- tissue_type.samples ---
tissue_type.samples
Tumor    514
Name: count, dtype: int64


--- primary_site ---
primary_site
Other and unspecified parts of tongue                                   126
Larynx                                                                  116
Other and ill-defined sites in lip, oral cavity and pharynx              70
Floor of mouth                                                           54
Tonsil                                                                   44
Other and unspecified parts of mouth                                     41
Base of tongue     